# CCA Model Input Diagnostics
## Diagnosing Distance-to-Distress Calibration Issues

This notebook systematically checks each input to the sovereign CCA model to identify
why d₂ values may differ from Gapen et al. (2008) benchmarks.

**Gapen et al. benchmarks:**
- d₂ range: 0.5 (stress) to 3.0 (calm)
- V_DCL / DB ratio: ~0.5–3.0 for typical EMs
- Vol_DCL: 20–76% (annualized)
- σ_A (implied): 15–45%
- V_A / DB ratio: ~1.2–3.0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import norm
from scipy.optimize import fsolve
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
# ============================================================
# LOAD DATA - adjust path as needed
# ============================================================
DATA_PATH = '../data/weekly_panel.csv'  # <-- ADJUST THIS PATH

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])

print(f"Shape: {df.shape}")
print(f"Countries: {df['country'].nunique()} -> {sorted(df['country'].unique())}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nColumns:\n{df.columns.tolist()}")

In [ ]:
# Quick look at raw values
key_cols = ['country', 'date', 'cds_spread_5Y', 'cds_spread_1Y', 
            'monetary_base_mn_localcurr', 'domestic_debt_bn_localcurr',
            'external_debt_mn_usd', 'domestic_rate_in_units', 
            'risk_free_rate', 'fx_rate']

df[key_cols].describe()

---
## 1. Unit Inspection

**Critical first check**: Are the units consistent across countries?
- `monetary_base_mn_localcurr`: millions of local currency
- `domestic_debt_bn_localcurr`: **billions** of local currency ⚠️ different unit!
- `external_debt_mn_usd`: millions of USD

We need everything in the same unit (e.g., millions of USD).

In [ ]:
# ============================================================
# CHECK 1: Raw magnitudes by country (latest available obs)
# ============================================================
latest = df.sort_values('date').groupby('country').last().reset_index()

print("="*90)
print("RAW VALUES AT LATEST OBSERVATION (check for unit issues)")
print("="*90)
cols_to_show = ['country', 'date', 'monetary_base_mn_localcurr', 
                'domestic_debt_bn_localcurr', 'external_debt_mn_usd', 'fx_rate']
print(latest[cols_to_show].to_string(index=False))

In [ ]:
# ============================================================
# UNIT HARMONIZATION: Convert everything to MILLIONS of USD
# ============================================================

# monetary_base: mn local curr -> mn USD
df['monetary_base_mn_usd'] = df['monetary_base_mn_localcurr'] / df['fx_rate']

# domestic_debt: bn local curr -> mn local curr -> mn USD
# ⚠️ NOTE: domestic_debt is in BILLIONS, monetary_base is in MILLIONS
df['domestic_debt_mn_usd'] = (df['domestic_debt_bn_localcurr'] * 1000) / df['fx_rate']

# external_debt: already in mn USD
df['external_debt_mn_usd_clean'] = df['external_debt_mn_usd']

print("Unit-converted columns created.")
print("\nSanity check - latest values in mn USD:")
latest2 = df.sort_values('date').groupby('country').last().reset_index()
print(latest2[['country', 'monetary_base_mn_usd', 'domestic_debt_mn_usd', 
               'external_debt_mn_usd_clean']].to_string(index=False))

---
## 2. Construct CCA Balance Sheet Components

Following Gapen et al. (2008):

**Local Currency Liabilities (LCL = V_DCL):**
$$V_{DCL} = \frac{\text{Monetary Base} + \text{Domestic Govt Debt}}{\text{FX Rate}}$$

**Distress Barrier (DB):**
$$DB = \text{ST External Debt} + \text{Interest} + 0.5 \times \text{LT External Debt}$$

⚠️ If we don't have ST/LT split, approximate as:
$$DB \approx 0.75 \times \text{Total External Debt}$$
(assuming ~1/3 is short-term, so ST + 0.5*LT ≈ 0.33 + 0.5*0.67 ≈ 0.67–0.75)

In [ ]:
# ============================================================
# CONSTRUCT V_DCL and DB
# ============================================================

# V_DCL = monetary_base + domestic_debt, all in mn USD
df['V_DCL_mn_usd'] = df['monetary_base_mn_usd'] + df['domestic_debt_mn_usd']

# DB approximation (adjust if you have ST/LT split)
DB_FRACTION = 0.75  # <-- ADJUST if you have actual ST/LT data
df['DB_mn_usd'] = DB_FRACTION * df['external_debt_mn_usd_clean']

# Key ratio
df['V_DCL_over_DB'] = df['V_DCL_mn_usd'] / df['DB_mn_usd']

print("V_DCL / DB ratio by country (latest obs):")
print("="*60)
latest3 = df.sort_values('date').groupby('country').last().reset_index()
ratio_df = latest3[['country', 'V_DCL_mn_usd', 'DB_mn_usd', 'V_DCL_over_DB']].copy()
ratio_df = ratio_df.sort_values('V_DCL_over_DB', ascending=False)
print(ratio_df.to_string(index=False))

print("\n" + "="*60)
print("BENCHMARK: Gapen et al. hypothetical = 82/100 = 0.82")
print("Typical range: 0.5 - 3.0")
print(f"Your median: {ratio_df['V_DCL_over_DB'].median():.2f}")
print(f"Your mean:   {ratio_df['V_DCL_over_DB'].mean():.2f}")
flagged = ratio_df[ratio_df['V_DCL_over_DB'] > 5]
if len(flagged) > 0:
    print(f"\n⚠️  {len(flagged)} countries with V_DCL/DB > 5 (likely inflated LCL):")
    print(flagged[['country', 'V_DCL_over_DB']].to_string(index=False))

In [ ]:
# ============================================================
# VISUAL: V_DCL / DB ratio distribution
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart of latest ratios
ax = axes[0]
ratio_sorted = ratio_df.sort_values('V_DCL_over_DB')
colors = ['red' if v > 5 else 'orange' if v > 3 else 'green' 
          for v in ratio_sorted['V_DCL_over_DB']]
ax.barh(ratio_sorted['country'], ratio_sorted['V_DCL_over_DB'], color=colors, alpha=0.7)
ax.axvline(x=0.82, color='blue', linestyle='--', label='Gapen benchmark (0.82)')
ax.axvline(x=3.0, color='red', linestyle='--', alpha=0.5, label='Upper reasonable (3.0)')
ax.set_xlabel('V_DCL / DB')
ax.set_title('V_DCL / DB Ratio by Country (latest obs)')
ax.legend()

# Components breakdown
ax = axes[1]
comp = latest3[['country', 'monetary_base_mn_usd', 'domestic_debt_mn_usd', 'DB_mn_usd']].copy()
comp = comp.sort_values('country')
# Show as log scale if magnitudes vary a lot
x = np.arange(len(comp))
w = 0.25
ax.bar(x - w, comp['monetary_base_mn_usd'], w, label='Mon. Base (mn USD)', alpha=0.7)
ax.bar(x, comp['domestic_debt_mn_usd'], w, label='Dom. Debt (mn USD)', alpha=0.7)
ax.bar(x + w, comp['DB_mn_usd'], w, label='DB (mn USD)', alpha=0.7, color='red')
ax.set_xticks(x)
ax.set_xticklabels(comp['country'], rotation=90, fontsize=8)
ax.set_yscale('log')
ax.set_ylabel('mn USD (log scale)')
ax.set_title('Balance Sheet Components')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 3. Diagnose: Which Component Dominates V_DCL?

If domestic_debt >> monetary_base, and domestic_debt >> external_debt,
then V_DCL is inflated and d₂ will be too high.

**Key question**: Is `domestic_debt_bn_localcurr` really *government* domestic debt,
or does it include private sector / total economy domestic debt?

In [ ]:
# ============================================================
# Decomposition: What fraction of V_DCL is domestic debt vs monetary base?
# ============================================================
decomp = latest3[['country']].copy()
decomp['mon_base_share'] = latest3['monetary_base_mn_usd'] / latest3['V_DCL_mn_usd']
decomp['dom_debt_share'] = latest3['domestic_debt_mn_usd'] / latest3['V_DCL_mn_usd']
decomp['dom_debt_to_ext_debt'] = latest3['domestic_debt_mn_usd'] / latest3['external_debt_mn_usd_clean']
decomp['V_DCL_over_DB'] = latest3['V_DCL_over_DB']
decomp = decomp.sort_values('dom_debt_to_ext_debt', ascending=False)

print("Component Decomposition:")
print("="*80)
print(decomp.to_string(index=False, float_format='{:.2f}'.format))
print("\nIf dom_debt_to_ext_debt >> 5, domestic debt likely includes non-sovereign items")
print("or external debt measure is too narrow.")

---
## 4. Vol_DCL Computation & Check

Vol_DCL is the annualized volatility of V_DCL in USD terms.

Main drivers:
- **FX volatility** (dominant for floating rates)
- **Quantity changes** in monetary base and domestic debt (dominant for pegs)

For weekly data: `Vol_DCL = std(weekly log returns) * sqrt(52)`

In [ ]:
# ============================================================
# COMPUTE Vol_DCL
# ============================================================

# Log returns of V_DCL
df = df.sort_values(['country', 'date'])
df['V_DCL_log_ret'] = df.groupby('country')['V_DCL_mn_usd'].transform(
    lambda x: np.log(x / x.shift(1))
)

# Also compute FX log returns for comparison
df['fx_log_ret'] = df.groupby('country')['fx_rate'].transform(
    lambda x: np.log(x / x.shift(1))
)

# Rolling annualized volatility (52-week window)
ROLL_WINDOW = 52
df['Vol_DCL_ann'] = df.groupby('country')['V_DCL_log_ret'].transform(
    lambda x: x.rolling(ROLL_WINDOW, min_periods=26).std() * np.sqrt(52)
)
df['Vol_FX_ann'] = df.groupby('country')['fx_log_ret'].transform(
    lambda x: x.rolling(ROLL_WINDOW, min_periods=26).std() * np.sqrt(52)
)

# Summary
vol_summary = df.groupby('country').agg(
    Vol_DCL_median=('Vol_DCL_ann', 'median'),
    Vol_DCL_mean=('Vol_DCL_ann', 'mean'),
    Vol_FX_median=('Vol_FX_ann', 'median'),
    Vol_FX_mean=('Vol_FX_ann', 'mean'),
).reset_index()

vol_summary = vol_summary.sort_values('Vol_DCL_median', ascending=False)
print("Annualized Volatilities:")
print("="*80)
print(vol_summary.to_string(index=False, float_format='{:.3f}'.format))
print("\nBENCHMARK: Gapen et al. Vol_DCL = 76% (high-vol example)")
print("Typical range for floating EM: 15-50%")
print("If Vol_DCL < 10%, asset vol will be very low -> d₂ inflated")

In [ ]:
# ============================================================
# VISUAL: Vol_DCL vs Vol_FX by country
# ============================================================
fig, ax = plt.subplots(figsize=(14, 6))

vs = vol_summary.sort_values('Vol_DCL_median')
x = np.arange(len(vs))
w = 0.35
ax.bar(x - w/2, vs['Vol_DCL_median'] * 100, w, label='Vol_DCL (annualized)', alpha=0.7)
ax.bar(x + w/2, vs['Vol_FX_median'] * 100, w, label='Vol_FX (annualized)', alpha=0.7, color='orange')
ax.axhline(y=76, color='red', linestyle='--', alpha=0.5, label='Gapen example (76%)')
ax.axhline(y=20, color='green', linestyle='--', alpha=0.5, label='Low end (20%)')
ax.set_xticks(x)
ax.set_xticklabels(vs['country'], rotation=90, fontsize=8)
ax.set_ylabel('Annualized Volatility (%)')
ax.set_title('Vol_DCL vs Vol_FX by Country')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Solve for Implied V_A and σ_A (Black-Scholes Inversion)

Solve the two-equation system:

$$V_{DCL} = V_A N(d_1) - DB \cdot e^{-r_f t} N(d_2)$$

$$\sigma_{DCL} \cdot V_{DCL} = \sigma_A \cdot V_A \cdot N(d_1)$$

for $(V_A, \sigma_A)$.

In [ ]:
# ============================================================
# CCA SOLVER
# ============================================================

def solve_cca(V_DCL, Vol_DCL, DB, rf, t=1.0):
    """
    Solve for implied sovereign asset value (V_A) and volatility (sigma_A)
    using the two-equation Black-Scholes system from Gapen et al. (2008).
    
    Returns: (V_A, sigma_A, d1, d2) or (NaN, NaN, NaN, NaN) if solver fails
    """
    if np.isnan(V_DCL) or np.isnan(Vol_DCL) or np.isnan(DB) or np.isnan(rf):
        return np.nan, np.nan, np.nan, np.nan
    if V_DCL <= 0 or Vol_DCL <= 0 or DB <= 0:
        return np.nan, np.nan, np.nan, np.nan
    
    def equations(params):
        V_A, sigma_A = params
        if V_A <= 0 or sigma_A <= 0:
            return [1e10, 1e10]
        
        d1 = (np.log(V_A / DB) + (rf + 0.5 * sigma_A**2) * t) / (sigma_A * np.sqrt(t))
        d2 = d1 - sigma_A * np.sqrt(t)
        
        # Equation 1: V_DCL = V_A * N(d1) - DB * exp(-rf*t) * N(d2)
        eq1 = V_A * norm.cdf(d1) - DB * np.exp(-rf * t) * norm.cdf(d2) - V_DCL
        
        # Equation 2: Vol_DCL * V_DCL = sigma_A * V_A * N(d1)
        eq2 = sigma_A * V_A * norm.cdf(d1) - Vol_DCL * V_DCL
        
        return [eq1, eq2]
    
    # Initial guess
    V_A_guess = V_DCL + DB
    sigma_A_guess = Vol_DCL * V_DCL / V_A_guess
    
    try:
        sol = fsolve(equations, [V_A_guess, sigma_A_guess], full_output=True)
        V_A, sigma_A = sol[0]
        info = sol[1]
        
        if V_A > 0 and sigma_A > 0 and sigma_A < 5:  # sanity bounds
            d1 = (np.log(V_A / DB) + (rf + 0.5 * sigma_A**2) * t) / (sigma_A * np.sqrt(t))
            d2 = d1 - sigma_A * np.sqrt(t)
            return V_A, sigma_A, d1, d2
    except:
        pass
    
    return np.nan, np.nan, np.nan, np.nan

print("CCA solver defined.")
print("\nTest with Gapen hypothetical: V_DCL=82, Vol_DCL=0.76, DB=100, rf=0.04")
test = solve_cca(82, 0.76, 100, 0.04)
print(f"  V_A = {test[0]:.1f} (expected ~175)")
print(f"  σ_A = {test[1]:.3f} (expected ~0.38)")
print(f"  d₂  = {test[3]:.2f} (expected ~1.4)")

In [ ]:
# ============================================================
# RUN CCA ON LATEST OBSERVATION FOR EACH COUNTRY
# ============================================================

# Use latest observation with valid Vol_DCL
latest_valid = df.dropna(subset=['Vol_DCL_ann']).sort_values('date').groupby('country').last().reset_index()

results = []
for _, row in latest_valid.iterrows():
    V_A, sigma_A, d1, d2 = solve_cca(
        V_DCL=row['V_DCL_mn_usd'],
        Vol_DCL=row['Vol_DCL_ann'],
        DB=row['DB_mn_usd'],
        rf=row['risk_free_rate'] / 100 if row['risk_free_rate'] > 1 else row['risk_free_rate'],
        t=1.0
    )
    results.append({
        'country': row['country'],
        'date': row['date'],
        'V_DCL': row['V_DCL_mn_usd'],
        'DB': row['DB_mn_usd'],
        'Vol_DCL': row['Vol_DCL_ann'],
        'rf': row['risk_free_rate'],
        'fx_rate': row['fx_rate'],
        'V_A_implied': V_A,
        'sigma_A': sigma_A,
        'd1': d1,
        'd2': d2,
        'V_A_over_DB': V_A / row['DB_mn_usd'] if not np.isnan(V_A) else np.nan,
        'V_DCL_over_DB': row['V_DCL_over_DB'],
        'cds_5y': row['cds_spread_5Y'],
    })

res_df = pd.DataFrame(results)
print("CCA Results (latest obs per country):")
print("="*120)
display_cols = ['country', 'V_DCL', 'DB', 'V_DCL_over_DB', 'Vol_DCL', 
                'V_A_implied', 'sigma_A', 'd2', 'cds_5y']
print(res_df[display_cols].sort_values('d2').to_string(
    index=False, 
    float_format='{:.3f}'.format
))

In [ ]:
# ============================================================
# DIAGNOSTIC SCATTER: d2 vs CDS spread
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

valid = res_df.dropna(subset=['d2', 'cds_5y'])

# d2 vs CDS
ax = axes[0]
ax.scatter(valid['d2'], valid['cds_5y'], alpha=0.7)
for _, r in valid.iterrows():
    ax.annotate(r['country'], (r['d2'], r['cds_5y']), fontsize=7)
ax.set_xlabel('d₂ (distance to distress)')
ax.set_ylabel('CDS 5Y (bps)')
ax.set_title('d₂ vs Market CDS Spread')
ax.axvline(x=1.4, color='red', linestyle='--', alpha=0.3, label='Gapen baseline')
ax.legend()

# sigma_A distribution
ax = axes[1]
ax.hist(valid['sigma_A'].dropna() * 100, bins=20, alpha=0.7, edgecolor='black')
ax.axvline(x=38, color='red', linestyle='--', label='Gapen (38%)')
ax.set_xlabel('Implied σ_A (%)')
ax.set_title('Distribution of Implied Asset Volatility')
ax.legend()

# V_A / DB distribution
ax = axes[2]
ax.hist(valid['V_A_over_DB'].dropna(), bins=20, alpha=0.7, edgecolor='black')
ax.axvline(x=1.75, color='red', linestyle='--', label='Gapen (1.75)')
ax.set_xlabel('V_A / DB')
ax.set_title('Distribution of Sovereign Asset / Distress Barrier')
ax.legend()

plt.tight_layout()
plt.show()

---
## 6. Sensitivity Analysis: What Drives d₂?

Test how d₂ changes when we vary:
1. The DB fraction (0.5 vs 0.75 vs 1.0 of external debt)
2. Excluding domestic debt from V_DCL (monetary base only)
3. Scaling domestic debt down (e.g., only central govt portion)

In [ ]:
# ============================================================
# SENSITIVITY: What if V_DCL = monetary base only (no domestic debt)?
# ============================================================

# This tests whether domestic_debt is the inflation source
results_monbase_only = []
for _, row in latest_valid.iterrows():
    V_DCL_alt = row['monetary_base_mn_usd']  # Only monetary base
    
    # Need Vol_DCL for this narrower measure
    # Approximate: use FX vol as proxy (since mon base in LC is sticky)
    Vol_alt = row['Vol_FX_ann'] if not np.isnan(row['Vol_FX_ann']) else row['Vol_DCL_ann']
    
    V_A, sigma_A, d1, d2 = solve_cca(
        V_DCL=V_DCL_alt,
        Vol_DCL=Vol_alt,
        DB=row['DB_mn_usd'],
        rf=row['risk_free_rate'] / 100 if row['risk_free_rate'] > 1 else row['risk_free_rate'],
        t=1.0
    )
    results_monbase_only.append({
        'country': row['country'],
        'd2_full_LCL': res_df[res_df['country']==row['country']]['d2'].values[0] if len(res_df[res_df['country']==row['country']]) > 0 else np.nan,
        'd2_monbase_only': d2,
        'V_DCL_full': row['V_DCL_mn_usd'],
        'V_DCL_monbase': V_DCL_alt,
        'inflation_factor': row['V_DCL_mn_usd'] / V_DCL_alt if V_DCL_alt > 0 else np.nan,
    })

sens_df = pd.DataFrame(results_monbase_only)
sens_df['d2_change'] = sens_df['d2_full_LCL'] - sens_df['d2_monbase_only']
print("Sensitivity: Full LCL vs Monetary Base Only")
print("="*90)
print(sens_df.sort_values('d2_change', ascending=False).to_string(
    index=False, float_format='{:.2f}'.format
))
print("\nPositive d2_change = domestic debt is inflating d₂")

In [ ]:
# ============================================================
# SENSITIVITY: Vary DB fraction
# ============================================================
print("Sensitivity: d₂ under different DB fractions")
print("="*80)

db_fracs = [0.50, 0.75, 1.00]
db_sens = []

for _, row in latest_valid.iterrows():
    entry = {'country': row['country']}
    for frac in db_fracs:
        DB_alt = frac * row['external_debt_mn_usd_clean']
        V_A, sigma_A, d1, d2 = solve_cca(
            V_DCL=row['V_DCL_mn_usd'],
            Vol_DCL=row['Vol_DCL_ann'],
            DB=DB_alt,
            rf=row['risk_free_rate'] / 100 if row['risk_free_rate'] > 1 else row['risk_free_rate'],
            t=1.0
        )
        entry[f'd2_DB={frac:.0%}'] = d2
    db_sens.append(entry)

db_sens_df = pd.DataFrame(db_sens)
print(db_sens_df.to_string(index=False, float_format='{:.2f}'.format))

---
## 7. Time Series Check: V_DCL and DB Over Time

Check for specific countries whether:
- V_DCL grows much faster than DB (domestic debt accumulation)
- Vol_DCL is suspiciously stable (pegged rate with no quantity adjustment)
- FX rate jumps are reflected properly

In [ ]:
# ============================================================
# TIME SERIES: Select countries for detailed inspection
# ============================================================

# Pick a few interesting countries (adjust as needed)
# Choose: one with high d2, one with reasonable d2, one problematic
countries_to_plot = df['country'].unique()[:6]  # First 6, or specify manually
# countries_to_plot = ['Brazil', 'Russia', 'Mexico', 'Saudi Arabia', 'Nigeria', 'Colombia']

fig, axes = plt.subplots(len(countries_to_plot), 3, figsize=(20, 4*len(countries_to_plot)))
if len(countries_to_plot) == 1:
    axes = axes.reshape(1, -1)

for i, country in enumerate(countries_to_plot):
    cdf = df[df['country'] == country].copy()
    
    # Panel 1: V_DCL vs DB over time
    ax = axes[i, 0]
    ax.plot(cdf['date'], cdf['V_DCL_mn_usd'], label='V_DCL', linewidth=1.5)
    ax.plot(cdf['date'], cdf['DB_mn_usd'], label='DB', linewidth=1.5, color='red')
    ax.plot(cdf['date'], cdf['monetary_base_mn_usd'], label='Mon. Base', 
            linewidth=1, linestyle='--', alpha=0.7)
    ax.set_title(f'{country}: V_DCL vs DB')
    ax.legend(fontsize=8)
    ax.set_ylabel('mn USD')
    
    # Panel 2: Vol_DCL and Vol_FX
    ax = axes[i, 1]
    ax.plot(cdf['date'], cdf['Vol_DCL_ann'] * 100, label='Vol_DCL', linewidth=1.5)
    ax.plot(cdf['date'], cdf['Vol_FX_ann'] * 100, label='Vol_FX', 
            linewidth=1, linestyle='--', alpha=0.7)
    ax.set_title(f'{country}: Volatilities')
    ax.set_ylabel('Annualized %')
    ax.legend(fontsize=8)
    
    # Panel 3: V_DCL / DB ratio
    ax = axes[i, 2]
    ax.plot(cdf['date'], cdf['V_DCL_over_DB'], linewidth=1.5, color='purple')
    ax.axhline(y=0.82, color='blue', linestyle='--', alpha=0.3)
    ax.set_title(f'{country}: V_DCL / DB Ratio')
    ax.set_ylabel('Ratio')

plt.tight_layout()
plt.show()

---
## 8. Risk-Free Rate Check

Verify `risk_free_rate` is in the right units (percent vs decimal).
- If stored as 4.5 (meaning 4.5%), need to divide by 100 for the formula
- If stored as 0.045, use directly

In [ ]:
# ============================================================
# RISK-FREE RATE CHECK
# ============================================================
print("Risk-free rate statistics:")
print(df.groupby('country')['risk_free_rate'].agg(['mean', 'min', 'max', 'last']).to_string())
print("\nIf values > 1, they are in PERCENT and need /100 in formulas.")
print("If values < 0.1, they are already in DECIMAL form.")

print("\n\nDomestic rate statistics:")
print(df.groupby('country')['domestic_rate_in_units'].agg(['mean', 'min', 'max', 'last']).to_string())

---
## 9. Summary Diagnosis

Run this after reviewing the outputs above to get a structured diagnosis.

In [ ]:
# ============================================================
# AUTOMATED DIAGNOSIS
# ============================================================
print("\n" + "="*80)
print("DIAGNOSTIC SUMMARY")
print("="*80)

valid_res = res_df.dropna(subset=['d2'])

# Check 1: d2 range
median_d2 = valid_res['d2'].median()
print(f"\n1. DISTANCE TO DISTRESS")
print(f"   Median d₂: {median_d2:.2f} (Gapen range: 0.5-3.0)")
if median_d2 > 4:
    print("   ⚠️  d₂ TOO HIGH - LCL likely inflated or DB too low")
elif median_d2 < 0.3:
    print("   ⚠️  d₂ TOO LOW - check if Vol_DCL is too high or DB too high")
else:
    print("   ✅ d₂ in reasonable range")

# Check 2: V_DCL/DB ratio
median_ratio = valid_res['V_DCL_over_DB'].median()
print(f"\n2. V_DCL / DB RATIO")
print(f"   Median: {median_ratio:.2f} (Gapen benchmark: 0.82)")
if median_ratio > 5:
    print("   ⚠️  RATIO TOO HIGH - domestic debt likely includes non-sovereign items")
    print("   or external debt measure is too narrow")

# Check 3: sigma_A
median_sigA = valid_res['sigma_A'].median()
print(f"\n3. IMPLIED ASSET VOLATILITY")
print(f"   Median σ_A: {median_sigA*100:.1f}% (Gapen range: 15-45%)")
if median_sigA < 0.05:
    print("   ⚠️  σ_A TOO LOW - Vol_DCL input may be too low")
    print("   This mechanically inflates d₂")

# Check 4: Component dominance
print(f"\n4. LCL COMPOSITION")
dom_share = (latest_valid['domestic_debt_mn_usd'] / latest_valid['V_DCL_mn_usd']).median()
print(f"   Median domestic debt share of V_DCL: {dom_share*100:.1f}%")
if dom_share > 0.9:
    print("   ⚠️  Domestic debt DOMINATES V_DCL")
    print("   Check if this is sovereign-only or includes broader economy")

# Check 5: Pegged countries
print(f"\n5. FX REGIME ISSUES")
low_vol_fx = vol_summary[vol_summary['Vol_FX_median'] < 0.03]
if len(low_vol_fx) > 0:
    print(f"   Countries with very low FX vol (<3%): {low_vol_fx['country'].tolist()}")
    print("   These are likely pegged - Vol_DCL may be understated")
    print("   Need to account for quantity changes in mon base")

print("\n" + "="*80)
print("POTENTIAL FIXES TO INVESTIGATE:")
print("="*80)
print("""
1. If V_DCL/DB >> 3: Check whether domestic_debt includes private sector
   debt or total economy debt (should be govt only)

2. If σ_A << 10%: Vol_DCL input may be too smooth. Check if monetary 
   base / domestic debt are forward-filled too aggressively

3. If DB seems too low: Check if external_debt captures all sovereign 
   FX obligations (including guaranteed debt, SOE debt)

4. Unit mismatch: monetary_base is in MILLIONS, domestic_debt in BILLIONS
   Make sure you multiply domestic_debt by 1000 before adding

5. For pegged countries: Vol_DCL ≈ Vol_FX understates true LCL volatility.
   Need to include quantity volatility of monetary base.
""")

---
## 10. Quick Fix Test: What DB Level Would Give Gapen-Like d₂?

For each country, find what DB would produce d₂ ≈ 1.5.
This tells us how far off our DB calibration is.

In [ ]:
# ============================================================
# REVERSE ENGINEER: What DB gives d2 ≈ 1.5?
# ============================================================
from scipy.optimize import brentq

target_d2 = 1.5

reverse_results = []
for _, row in latest_valid.iterrows():
    V_DCL = row['V_DCL_mn_usd']
    Vol_DCL = row['Vol_DCL_ann']
    rf = row['risk_free_rate'] / 100 if row['risk_free_rate'] > 1 else row['risk_free_rate']
    actual_DB = row['DB_mn_usd']
    
    if np.isnan(V_DCL) or np.isnan(Vol_DCL) or V_DCL <= 0 or Vol_DCL <= 0:
        continue
    
    def d2_for_DB(DB_test):
        _, _, _, d2 = solve_cca(V_DCL, Vol_DCL, DB_test, rf)
        if np.isnan(d2):
            return 10  # push away
        return d2 - target_d2
    
    # Search for DB that gives d2 = 1.5
    try:
        # Try a wide range
        DB_target = brentq(d2_for_DB, actual_DB * 0.01, V_DCL * 10, maxiter=200)
        reverse_results.append({
            'country': row['country'],
            'actual_DB': actual_DB,
            'DB_for_d2_1.5': DB_target,
            'DB_ratio': DB_target / actual_DB,
            'ext_debt': row['external_debt_mn_usd_clean'],
            'needed_frac': DB_target / row['external_debt_mn_usd_clean'] if row['external_debt_mn_usd_clean'] > 0 else np.nan,
        })
    except:
        reverse_results.append({
            'country': row['country'],
            'actual_DB': actual_DB,
            'DB_for_d2_1.5': np.nan,
            'DB_ratio': np.nan,
            'ext_debt': row['external_debt_mn_usd_clean'],
            'needed_frac': np.nan,
        })

rev_df = pd.DataFrame(reverse_results)
print(f"What DB would give d₂ = {target_d2}?")
print("="*80)
print(rev_df.sort_values('DB_ratio', ascending=False).to_string(
    index=False, float_format='{:.2f}'.format
))
print("\nDB_ratio > 3: Your DB is WAY too low (LCL dominates)")
print("DB_ratio ≈ 1: Your calibration is close")
print("needed_frac > 1: Even using 100% of ext debt as DB isn't enough")